# Study Context

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from study_context import ExperimentContext, experiment_context_summary, pipeline_for_accession_list

load_dotenv()
from shared.repo import REPO_ROOT

ROOT = REPO_ROOT

## Config

In [ ]:
# --- config ---
READ_FROM_CACHE = False
SAMPLE_SIZE = None  # set to int to limit accessions for testing, e.g. SAMPLE_SIZE = 5

ACCESSIONS_PATH = ROOT / "output/metadata/datasets.csv"
OUTPUT_PATH = ROOT / "output/context/contexts.jsonl"

## Fetch / Load

In [ ]:
if READ_FROM_CACHE and OUTPUT_PATH.exists():
    with open(OUTPUT_PATH) as f:
        contexts = [ExperimentContext.model_validate_json(line) for line in f if line.strip()]
    print(f"Loaded {len(contexts)} contexts from cache.")
else:
    data = pd.read_csv(ACCESSIONS_PATH)
    if SAMPLE_SIZE:
        data = data.sample(n=SAMPLE_SIZE, random_state=42)
    accessions = data["srx_accession"].tolist()
    print(f"Fetching context for {len(accessions)} accessions...")
    contexts = pipeline_for_accession_list(accessions)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_PATH, "w") as f:
        for ctx in contexts:
            f.write(ctx.model_dump_json() + "\n")
    print(f"Saved {len(contexts)} contexts to {OUTPUT_PATH}")

## Basic Counts

In [ ]:
# basic counts: check for missing, extra, and duplicate accessions
source_accessions = pd.read_csv(ACCESSIONS_PATH)["srx_accession"].tolist()
loaded_accessions = [ctx.accession for ctx in contexts]

missing = set(source_accessions) - set(loaded_accessions)
extra = set(loaded_accessions) - set(source_accessions)
dupes = [a for a in loaded_accessions if loaded_accessions.count(a) > 1]

print(f"Source accessions:     {len(source_accessions)}")
print(f"Loaded contexts:       {len(contexts)}")
print(f"Missing from contexts: {len(missing)}")
print(f"Extra in contexts:     {len(extra)}")
print(f"Duplicates:            {len(set(dupes))}")

if missing:
    print("\nMissing accessions:", sorted(missing))

## Field Coverage

In [ ]:
# field coverage
n = len(contexts)

fields = {
    "studyDescription": sum(1 for c in contexts if c.study and c.study.studyDescription),
    "pubmedAbstract": sum(1 for c in contexts if c.study and c.study.pubmedAbstract),
    "biological.tissueType": sum(1 for c in contexts if c.biological.tissueType),
    "biological.cellType": sum(1 for c in contexts if c.biological.cellType),
    "biological.sampleAttributes": sum(1 for c in contexts if c.biological.sampleAttributes),
    "technical.libraryStrategy": sum(1 for c in contexts if c.technical.libraryStrategy),
    "technical.libraryConstructionProtocol": sum(1 for c in contexts if c.technical.libraryConstructionProtocol),
}

pd.DataFrame(
    {
        "field": fields.keys(),
        "populated": fields.values(),
        "missing": [n - v for v in fields.values()],
        "pct": [f"{v / n * 100:.1f}%" for v in fields.values()],
    }
)

## Warnings

In [ ]:
# warnings
warned = [(ctx.accession, w) for ctx in contexts for w in ctx.warnings]

print(f"Records with warnings: {sum(1 for c in contexts if c.warnings)} / {n}")
print(f"Total warning entries: {len(warned)}\n")

if warned:
    warning_types = pd.Series([w for _, w in warned]).value_counts()
    print("Warning type breakdown:")
    print(warning_types.to_string())
    print("\nAffected accessions:")
    for acc, w in warned:
        print(f"  {acc}: {w}")

## Distributions

In [ ]:
# distributions
print("Library strategy:")
print(pd.Series([c.technical.libraryStrategy for c in contexts]).value_counts().to_string())

print("\nSpecies:")
print(pd.Series([c.biological.scientificName for c in contexts]).value_counts().to_string())

print("\nTissue type (top 20):")
print(pd.Series([c.biological.tissueType for c in contexts]).value_counts().head(20).to_string())

## Spot Checks

In [ ]:
# spot checks
def print_ctx(ctx: ExperimentContext) -> None:
    print(f"Accession:         {ctx.accession}")
    print(f"Experiment title:  {ctx.experimentTitle}")
    print(f"Species:           {ctx.biological.scientificName}")
    print(f"Tissue:            {ctx.biological.tissueType}")
    print(f"Library strategy:  {ctx.technical.libraryStrategy}")
    print(f"Study description: {(ctx.study.studyDescription or '')[:200] if ctx.study else ''}")
    print(f"GEO accession:     {ctx.study.geoAccession if ctx.study else None}")
    print(f"PubMed abstract:   {(ctx.study.pubmedAbstract or '')[:200] if ctx.study else ''}")
    print(f"Warnings:          {ctx.warnings}")
    print()


print("First 3 records:")
for ctx in contexts[:3]:
    print_ctx(ctx)

no_abstract = [c for c in contexts if not (c.study and c.study.pubmedAbstract)]
print(f"Records without pubmedAbstract: {len(no_abstract)}")
print("\nFirst 5 without abstract:")
for ctx in no_abstract[:5]:
    print_ctx(ctx)

## Lookup Helpers

In [ ]:
def get_context(accession: str) -> ExperimentContext | None:
    return next((ctx for ctx in contexts if ctx.accession == accession), None)


# example
ctx = get_context("SRX17412841")
print(experiment_context_summary(ctx))

## Text Length

In [ ]:
# combined text length (study description + pubmed abstract)
CHAR_LIMIT = 300

combined_lengths = [
    len(ctx.study.studyDescription or "") + len(ctx.study.pubmedAbstract or "") if ctx.study else 0 for ctx in contexts
]
below = sum(l < CHAR_LIMIT for l in combined_lengths)
print(
    f"{below} of {n} records ({below / n * 100:.1f}%) have combined studyDescription + pubmedAbstract < {CHAR_LIMIT} chars"
)